# Laboratorio 5.1: Regresión con PyTorch (ATP Tennis)

**Dataset:** ATP matches hasta 2022  
**Tarea:** Predecir la duración del partido (`minutes`) a partir de características del partido.  
**Tipo de problema:** Regresión  

| Concepto | Fuente |
|---|---|
| `nn.Sequential`, `nn.Module`, optimizadores | `02-pytorch_nn.ipynb` |
| `Dataset`, `DataLoader` | `03-pytorch_datasets.ipynb` |
| Checkpoints, TorchScript | `04-pytorch_save.ipynb` |

---
## Fase 1: Carga y Exploración del Dataset

El dataset tiene 188 161 partidos ATP con 49 columnas.  
Predecimos `minutes` (duración) a partir de: `surface`, `best_of`, `winner_rank`, `loser_rank`, `w_ace`, `l_ace`.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
import os
%matplotlib inline

df = pd.read_csv('../Datasets/1_ATP_matches_D/atp_matches_till_2022.csv')
print(f"Filas × Columnas: {df.shape}")
df[['surface','best_of','winner_rank','loser_rank','w_ace','l_ace','minutes']].describe()


### 1.1 Distribución del target (`minutes`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['minutes'].dropna().plot.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de minutes'); axes[0].set_xlabel('Minutos')
df['minutes'].dropna().plot.box(ax=axes[1])
axes[1].set_title('Boxplot de minutes')
plt.tight_layout(); plt.show()
print(f"Duración media: {df['minutes'].mean():.1f} min | Mediana: {df['minutes'].median():.1f} min")


---
## Fase 2: Preprocesamiento

Seleccionamos 6 features, codificamos `surface` con `pd.factorize`, eliminamos filas con `NaN` y separamos en train (70%), val (15%), test (15%).

In [ ]:
FEATURES = ['surface', 'best_of', 'winner_rank', 'loser_rank', 'w_ace', 'l_ace']
TARGET    = 'minutes'

df_clean = df[FEATURES + [TARGET]].copy()
df_clean['surface'] = pd.factorize(df_clean['surface'])[0]   # Hard→0, Clay→1, Grass→2, Carpet→3
df_clean = df_clean.dropna()
print(f"Filas tras limpiar NaN: {len(df_clean):,}")

X = df_clean[FEATURES].values.astype(np.float32)
y = df_clean[TARGET].values.astype(np.float32).reshape(-1, 1)

# Split 70/15/15
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
n_train = int(0.70 * len(X))
n_val   = int(0.15 * len(X))

idx_tr  = idx[:n_train]
idx_val = idx[n_train:n_train + n_val]
idx_te  = idx[n_train + n_val:]

X_tr, y_tr   = X[idx_tr],  y[idx_tr]
X_val, y_val = X[idx_val], y[idx_val]
X_te, y_te   = X[idx_te],  y[idx_te]

# Normalizar features con estadísticas del train
mean_X = X_tr.mean(axis=0); std_X = X_tr.std(axis=0) + 1e-8
mean_y = y_tr.mean();        std_y = y_tr.std()  + 1e-8

X_tr  = (X_tr  - mean_X) / std_X
X_val = (X_val - mean_X) / std_X
X_te  = (X_te  - mean_X) / std_X

y_tr  = (y_tr  - mean_y) / std_y
y_val = (y_val - mean_y) / std_y
y_te  = (y_te  - mean_y) / std_y

print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_te.shape}")


---
## Fase 3: Dataset y DataLoader

In [ ]:
class ATPDataset(Dataset):
    """Dataset para ATP: normalización ya aplicada externamente."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):  return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 512
train_ds = ATPDataset(X_tr,  y_tr)
val_ds   = ATPDataset(X_val, y_val)
test_ds  = ATPDataset(X_te,  y_te)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f"Batch X: {xb.shape} | Batch y: {yb.shape}")


---
## Fase 4: Modelo — MLP Regresión

Para regresión la capa de salida tiene **1 neurona sin activación** y la función de pérdida es `MSELoss`.

In [ ]:
class MLPReg(nn.Module):
    """MLP para regresión con BatchNorm y Dropout."""
    def __init__(self, in_features=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),           nn.ReLU(),
            nn.Linear(32, 1)             # sin activación → regresión
        )

    def forward(self, x): return self.net(x)

modelo = MLPReg(in_features=len(FEATURES))
total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {total_params:,}")
print(modelo)


## Fase 4.1: Entrenamiento

In [ ]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS      = 50
LR          = 1e-3
BEST_PATH   = './checkpoints/atp_best.pth'
RESUME_PATH = './checkpoints/atp_resume.pth'
os.makedirs('./checkpoints', exist_ok=True)

modelo = MLPReg(in_features=len(FEATURES)).to(DEVICE)
optimizer = torch.optim.Adam(modelo.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
criterion = nn.MSELoss()

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

# ── Reanudar si existe backup ──────────────────────────────────
if os.path.exists(RESUME_PATH):
    ckpt = torch.load(RESUME_PATH, weights_only=False)
    modelo.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    history       = ckpt['history']
    print(f"▶ Reanudando desde época {start_epoch}")
else:
    start_epoch = 1
    print("▶ Entrenamiento desde cero.")

for epoch in range(start_epoch, EPOCHS + 1):
    # Entrenamiento
    modelo.train()
    t_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(modelo(xb), yb)
        loss.backward(); optimizer.step()
        t_loss += loss.item() * xb.size(0)
    scheduler.step()

    # Validación
    modelo.eval()
    v_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            v_loss += criterion(modelo(xb), yb).item() * xb.size(0)

    tl = t_loss / len(train_ds)
    vl = v_loss / len(val_ds)
    history['train_loss'].append(tl); history['val_loss'].append(vl)

    # Best model
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(modelo.state_dict(), BEST_PATH)
        print(f"   ✓ Mejor modelo guardado (val_loss={vl:.4f})")

    # Backup cada 10 épocas
    if epoch % 10 == 0:
        torch.save({'epoch': epoch, 'model_state': modelo.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'scheduler_state': scheduler.state_dict(),
                    'history': history, 'best_val_loss': best_val_loss}, RESUME_PATH)
        print(f"   💾 Backup (época {epoch}/{EPOCHS})")

    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={tl:.4f} | val_loss={vl:.4f}")

modelo.load_state_dict(torch.load(BEST_PATH, weights_only=True))
print(f"\n✅ Entrenamiento finalizado. Mejor val_loss: {best_val_loss:.4f}")


## Fase 4.2: Curvas de entrenamiento

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['val_loss'],   label='val_loss')
plt.xlabel('Época'); plt.ylabel('MSE (normalizado)')
plt.title('Curvas de pérdida — ATP Regresión')
plt.legend(); plt.tight_layout(); plt.show()


---
## Fase 5: Guardar / Cargar Modelos

### 5.1 `state_dict` (recomendado)

In [ ]:
torch.save(modelo.state_dict(), './checkpoints/atp_demo.pth')

model_cargado = MLPReg(in_features=len(FEATURES))
model_cargado.load_state_dict(torch.load('./checkpoints/atp_demo.pth', weights_only=True))
model_cargado.eval()
print("state_dict cargado correctamente ✓")


### 5.2 Modelo completo

In [ ]:
torch.save(modelo, './checkpoints/atp_completo.pt')
model_full = torch.load('./checkpoints/atp_completo.pt', weights_only=False)
model_full.eval()
print("Modelo completo cargado ✓")


### 5.3 TorchScript

In [ ]:
modelo.cpu().eval()
scripted = torch.jit.script(modelo)
scripted.save('./checkpoints/atp_scripted.pt')
loaded_ts = torch.jit.load('./checkpoints/atp_scripted.pt')
demo_in   = torch.randn(1, len(FEATURES))
print("TorchScript ✓ | output shape:", loaded_ts(demo_in).shape)
modelo.to(DEVICE)


---
## Fase 6: Evaluación Final en Test

### 6.1 MAE y RMSE (en escala original de minutos)

In [ ]:
best_model = MLPReg(in_features=len(FEATURES))
best_model.load_state_dict(torch.load(BEST_PATH, weights_only=True))
best_model.eval()
best_model.to(DEVICE)

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        pred = best_model(xb).cpu().numpy()
        all_preds.append(pred)
        all_true.append(yb.numpy())

preds_norm = np.concatenate(all_preds)
trues_norm = np.concatenate(all_true)

# Deshacer normalización del target
preds_min = preds_norm * std_y + mean_y
trues_min = trues_norm * std_y + mean_y

mae  = np.abs(preds_min - trues_min).mean()
rmse = np.sqrt(((preds_min - trues_min) ** 2).mean())
print(f"✅ MAE  en Test: {mae:.2f} minutos")
print(f"✅ RMSE en Test: {rmse:.2f} minutos")


### 6.2 Predicción vs Real (muestra aleatoria)

In [ ]:
rng2 = np.random.default_rng(7)
idx_sample = rng2.choice(len(trues_min), 20, replace=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trues_min[idx_sample],  'o-', label='Real')
ax.plot(preds_min[idx_sample],  's--', label='Predicción')
ax.set_xlabel('Muestra'); ax.set_ylabel('Minutos')
ax.set_title('Predicción vs Real — ATP Test Set (20 muestras)')
ax.legend(); plt.tight_layout(); plt.show()


### 6.3 Predicción manual

Ingresa los datos de un partido para predecir su duración.  
`surface`: 0=Hard, 1=Clay, 2=Grass, 3=Carpet

In [ ]:
partido = {
    'surface':     0,    # Hard
    'best_of':     3,
    'winner_rank': 5,
    'loser_rank':  122,
    'w_ace':       8,
    'l_ace':       3,
}

x_raw = np.array([[partido[f] for f in FEATURES]], dtype=np.float32)
x_norm = (x_raw - mean_X) / std_X
x_in   = torch.tensor(x_norm, dtype=torch.float32).to(DEVICE)

best_model.eval()
with torch.no_grad():
    pred_norm = best_model(x_in).item()
pred_min = pred_norm * std_y + mean_y
print(f"Predicción de duración: {pred_min:.0f} minutos (~{pred_min/60:.1f} horas)")
